# Bell-State Effective Qubits (BSEQ)

In [1]:
# (C) Copyright IBM 2026.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

In [ ]:
import time
import numpy as np
from qiskit import QuantumCircuit
from qiskit.result import marginal_counts, sampled_expectation_value
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import Sampler
import rustworkx as rx

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

In [ ]:
import json
from datetime import datetime

def default(obj):
    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat(),
        }
    raise TypeError(f"Cannot serialize {type(obj)}")

def object_hook(obj):
    if obj.get("__type__") == "datetime":
        return datetime.fromisoformat(obj["value"])
    return obj

data = {
    "name": "test",
    "created": datetime.now(),
}

In [ ]:
service = QiskitRuntimeService()

In [ ]:
backend = service.backend("ibm_chicago")

In [ ]:
class GraphColoring:
    """A simple class containing graph coloring data"""

    def __init__(self, num_nodes, edge_color_map, edge_index_map):
        self.edge_color_map = edge_color_map
        self.edge_index_map = edge_index_map
        self.num_colors = max(edge_color_map.values()) + 1
        self.num_nodes = num_nodes

In [ ]:
def generate_chsh_circuit_sets(coloring):
    """Generate CHSH circuits from a coloring

    Parameters:
        coloring (GraphColoring): Coloring from a backend

    Returns:
        list: Nested list of QuantumCircuit objects
    """
    num_qubits = coloring.num_nodes
    circuits = []
    # For each coloring, generate a set of CHSH pairs (Bell pairs plus an Ry(pi/4)) on each
    # edge of the coloring.  Measurement register is twice the number of qubit pairs in the
    # coloring
    for counter in range(coloring.num_colors):
        num_qubit_pairs = len(
            {key for key, val in coloring.edge_color_map.items() if val == counter}
        )
        qc = QuantumCircuit(num_qubits, 2 * num_qubit_pairs)
        for edge_idx in (
            key for key, val in coloring.edge_color_map.items() if val == counter
        ):
            edge = (
                coloring.edge_index_map[edge_idx][0],
                coloring.edge_index_map[edge_idx][1],
            )
            # For each edge in the color set perform a CHSH experiment at the optimal value
            qc.h(edge[0])
            qc.cx(*edge)
            qc.ry(np.pi / 4, edge[0])
        circuits.append(qc)

    exp_sets = []
    # For each coloring circuit, generate 4 new circuits with the required post-rotation operators
    # and measurements appended
    for counter, circ in enumerate(circuits):
        meas_circuits = []
        # Need to create a circuit for each measurement basis.  This amounts to appending
        # a H gate to the qubits with an X-basis measurement
        for basis in ["ZZ", "ZX", "XZ", "XX"]:
            temp_qc = circ.copy()
            meas_counter = 0
            for edge_idx in (
                key for key, val in coloring.edge_color_map.items() if val == counter
            ):
                edge = (
                    coloring.edge_index_map[edge_idx][0],
                    coloring.edge_index_map[edge_idx][1],
                )
                for idx, oper in enumerate(basis[::-1]):
                    if oper == "X":
                        temp_qc.h(edge[idx])
                temp_qc.measure(edge, [meas_counter, meas_counter + 1])
                meas_counter += 2
            meas_circuits.append(temp_qc)
        exp_sets.append(meas_circuits)

    return exp_sets

In [ ]:
def chsh_subgraph(jobs, coloring):
    """Return subgraph where all edges correspond to qubit
    pairs that violate the CHSH inequality

    Parameters:
        jobs (list): A list of jobs to process
        coloring (GraphColoring): The backend coloring

    Returns:
        PyGraph: Graph for the CHSH subgraph
    """
    good_edges = []
    for job_idx, job in enumerate(jobs):
        num_meas_pairs = len(
            {key for key, val in coloring.edge_color_map.items() if val == job_idx}
        )
        exp_vals = np.zeros(num_meas_pairs, dtype=float)
        for idx in range(4):
            counts = job.result()[idx].data.c.get_counts()
            for pair in range(num_meas_pairs):
                sub_counts = marginal_counts(counts, [2 * pair, 2 * pair + 1])
                exp_val = sampled_expectation_value(sub_counts, "ZZ")
                exp_vals[pair] += exp_val if idx != 2 else -exp_val
        for idx, edge_idx in enumerate(
            key for key, val in coloring.edge_color_map.items() if val == job_idx
        ):
            edge = (
                coloring.edge_index_map[edge_idx][0],
                coloring.edge_index_map[edge_idx][1],
            )
            if exp_vals[idx] > 2:
                good_edges.append((edge, float(exp_vals[idx])))
    good_graph = rx.PyGraph(multigraph=False)
    good_graph.add_nodes_from(list(range(coloring.num_nodes)))
    for edge in good_edges:
        good_graph.add_edge(*edge[0], 1)
    return good_graph, good_edges


def largest_connected_component(good_graph):
    cc = rx.connected_components(good_graph)
    largest_cc = cc[np.argmax([len(g) for g in cc])]
    return largest_cc


def graph_coloring(graph):
    """Graph coloring for a backend entangling gate topology

    Parameters:
        graph : Graph to be colored

    Returns:
        GraphColoring: Coloring object
    """
    # Get the graph of the coupling map
    # Got to undirected graph for coloring
    undirected_graph = graph.to_undirected(multigraph=False)
    # Graphs are bipartite so use that feature to prevent extra colors from greedy search
    edge_color_map = rx.graph_bipartite_edge_color(undirected_graph)
    # Get the index of the edges
    edge_index_map = undirected_graph.edge_index_map()
    return GraphColoring(graph.num_nodes(), edge_color_map, edge_index_map)

In [ ]:
classical_time = 0
st = time.perf_counter()
coloring = graph_coloring(backend.coupling_map.graph)
exp_sets = generate_chsh_circuit_sets(coloring)
pm = generate_preset_pass_manager(1, backend)
trans_exp_sets = [pm.run(circ_set) for circ_set in exp_sets]
ft = time.perf_counter()
classical_time += ft - st

In [ ]:
shots = 1024

In [ ]:
sampler = Sampler(backend)
jobs = [sampler.run(circ_set, shots=shots) for circ_set in trans_exp_sets]

In [ ]:
good_graph, good_edges = chsh_subgraph(jobs, coloring)
st = time.perf_counter()
largest_cc = largest_connected_component(good_graph)
largest_size = len(largest_cc)
# --- get weighted size here
largest_list = list(largest_cc)
mapping = {}
for kk in range(largest_size):
    mapping[largest_list[kk]] = kk

mapped_edges = []
for edge in backend.coupling_map:
    if edge[0] in largest_cc and edge[1] in largest_cc:
        new_edge = (mapping[edge[0]], mapping[edge[1]], 1)
        flipped_edge = (mapping[edge[1]], mapping[edge[0]], 1)
        # Check if flipped edge is already in list
        if flipped_edge not in mapped_edges:
            mapped_edges.append(new_edge)
subgraph = rx.PyGraph()
subgraph.add_nodes_from(range(largest_size))
subgraph.add_edges_from(mapped_edges)
avg_good_deg = np.mean([subgraph.degree(kk) for kk in range(subgraph.num_nodes())])
ft = time.perf_counter()
classical_time += ft - st

In [ ]:
results = {'name': 'bseq', 'backend': backend.name, 'datetime': str(jobs[0].creation_date),
           'backend_properties': json.dumps(backend.properties().to_dict(), default=default),
           'shots': shots, 'quantum_runtime': sum(mm.usage() for mm in jobs), 'classical_runtime': classical_time,
           'result': largest_size, 'largest_connected_component':largest_list, 'good_edges':good_edges}

In [ ]:
with open("bseq_results.json", "w") as f:
        json.dump(results, f)